In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window

In [2]:
# holidays in NYC from between 2023 and 2025
holidays = ["2023-01-02", "2023-01-16", "2023-02-13", "2023-02-20", "2023-05-29", "2023-06-19", "2023-07-04", "2023-09-04", "2023-10-09", "2023-11-10", "2023-11-23", "2023-12-25", "2024-01-01", "2024-01-15", "2024-02-12", "2024-02-19", "2024-05-27", "2024-06-19", "2024-07-04", "2024-09-02", "2024-10-14", "2024-11-11", "2024-11-28", "2024-12-25", "2025-01-01", "2025-01-20", "2025-02-12", "2025-02-17", "2025-05-26", "2025-06-19", "2025-07-04", "2025-09-01", "2025-10-13", "2025-11-11", "2025-11-27", "2025-12-25"]

In [3]:
spark = SparkSession.builder.config("spark.driver.memory", "6g").config("spark.driver.maxResultSize", "2g").master("local[*]").appName("bad_weather_driving_behavior").getOrCreate()
base_path = "/home/jovyan/work"

In [4]:
df_weather = spark.read.parquet(f"{base_path}/data/weather/cleaned_weather")
df_weather.printSchema()

root
 |-- STATION: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- LATITUDE: string (nullable = true)
 |-- LONGITUDE: string (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- HourlyDryBulbTemperature: string (nullable = true)
 |-- HourlyPrecipitation: string (nullable = true)
 |-- HourlyPresentWeatherType: string (nullable = true)
 |-- DailyAverageDryBulbTemperature: string (nullable = true)
 |-- DailyMaximumDryBulbTemperature: string (nullable = true)
 |-- DailyMinimumDryBulbTemperature: string (nullable = true)
 |-- DailyPrecipitation: string (nullable = true)
 |-- DailyWeather: string (nullable = true)
 |-- Sunrise: string (nullable = true)
 |-- Sunset: string (nullable = true)



In [48]:
# remove rows with NULL values (according to NOAA LCD documentation: NULL != no precipitation)
df_weather = df_weather \
    .select("STATION", "LATITUDE", "LONGITUDE", "DATE", "HourlyPrecipitation", "HourlyPresentWeatherType") \
    .filter(col("HourlyPrecipitation").isNotNull())

In [49]:
# remove rows where it is unclear if its snows or rains
df_weather = df_weather \
    .filter(~coalesce(col("HourlyPresentWeatherType").contains("SN") & col("HourlyPresentWeatherType").contains("RA"), lit(False))) \
    .withColumn("precipitation_type", when(col("HourlyPresentWeatherType").contains("RA") | col("HourlyPresentWeatherType").contains("DZ"), "rain").when(col("HourlyPresentWeatherType").contains("SN"), "snow").otherwise("dry"))

In [50]:
# extract hour and date from timestamp
df_weather = df_weather \
    .withColumn("hour", hour(col("DATE"))) \
    .withColumn("day", to_date(col("DATE")))

df_weather.printSchema()

root
 |-- STATION: string (nullable = true)
 |-- LATITUDE: string (nullable = true)
 |-- LONGITUDE: string (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- HourlyPrecipitation: string (nullable = true)
 |-- HourlyPresentWeatherType: string (nullable = true)
 |-- precipitation_type: string (nullable = false)
 |-- hour: integer (nullable = true)
 |-- day: date (nullable = true)



In [51]:
df_weather.show(5, False)

+-----------+--------+---------+-------------------+-------------------+------------------------+------------------+----+----------+
|STATION    |LATITUDE|LONGITUDE|DATE               |HourlyPrecipitation|HourlyPresentWeatherType|precipitation_type|hour|day       |
+-----------+--------+---------+-------------------+-------------------+------------------------+------------------+----+----------+
|USW00094789|40.6386 |-73.7622 |2024-01-01 23:51:00|0.0                |NULL                    |dry               |23  |2024-01-01|
|USW00094789|40.6386 |-73.7622 |2024-01-01 22:51:00|0.0                |NULL                    |dry               |22  |2024-01-01|
|USW00094789|40.6386 |-73.7622 |2024-01-01 21:51:00|0.0                |NULL                    |dry               |21  |2024-01-01|
|USW00094789|40.6386 |-73.7622 |2024-01-01 20:51:00|0.0                |NULL                    |dry               |20  |2024-01-01|
|USW00094789|40.6386 |-73.7622 |2024-01-01 19:51:00|0.0              

In [52]:
df_speeds = spark.read.parquet(f"{base_path}/data/traffic_speeds/cleaned_traffic_speeds")
df_speeds.printSchema()

root
 |-- ID: string (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- SPEED: string (nullable = true)
 |-- TRAVEL_TIME: string (nullable = true)
 |-- LINK_POINTS: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- BOROUGH: string (nullable = true)



In [53]:
# remove holidays, because a different driving behaviour is expected
# add day-type (weekday, saturday, sunday)
df_speeds = df_speeds \
    .filter(~col("DATE").isin(holidays)) \
    .withColumn("day_type", when(weekday(col("DATE")) == 5, "sat").when(weekday(col("DATE")) == 6, "sun").otherwise("weekday"))

In [54]:
# extract speed sensors
df_sensors = df_speeds \
    .select("ID", "LINK_POINTS", "BOROUGH") \
    .distinct()

In [55]:
df_sensors.printSchema()

root
 |-- ID: string (nullable = true)
 |-- LINK_POINTS: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- BOROUGH: string (nullable = true)



In [57]:
# calculate center of sensor routes
df_sensors = df_sensors \
    .withColumn("lats", expr("transform(LINK_POINTS, x -> cast(split(x, ',')[0] as double))")) \
    .withColumn("lons", expr("transform(LINK_POINTS, x -> cast(split(x, ',')[1] as double))")) \
    .withColumn("center_lat", expr("aggregate(lats, 0D, (acc, x) -> acc + x) / size(lats)")) \
    .withColumn("center_lon", expr("aggregate(lons, 0D, (acc, x) -> acc + x) / size(lons)"))

In [58]:
# extract weather stations
df_stations = df_weather \
    .select("STATION", "LATITUDE", "LONGITUDE") \
    .distinct() \
    .withColumn("lat_station", col("LATITUDE").cast("double")) \
    .withColumn("lon_station", col("LONGITUDE").cast("double"))

In [59]:
df_stations.show(truncate=False)

+-----------+--------+---------+-----------+-----------+
|STATION    |LATITUDE|LONGITUDE|lat_station|lon_station|
+-----------+--------+---------+-----------+-----------+
|USW00014732|40.7792 |-73.88   |40.7792    |-73.88     |
|USW00014734|40.6825 |-74.1694 |40.6825    |-74.1694   |
|USW00094789|40.6386 |-73.7622 |40.6386    |-73.7622   |
|USW00094728|40.77898|-73.96925|40.77898   |-73.96925  |
|USW00094741|40.85   |-74.06139|40.85      |-74.06139  |
+-----------+--------+---------+-----------+-----------+



In [60]:
# join sensors and stations
df_sensors_stations = df_sensors.crossJoin(broadcast(df_stations.select("STATION", "lat_station", "lon_station")))

In [61]:
# calculate distance(in km) between each sensor and each station
R = 6371.0

df_sensors_stations = df_sensors_stations \
    .withColumn("dist_km", 2 * R * asin(sqrt(pow(sin(radians(col("lat_station") - col("center_lat")) / 2), 2) + cos(radians(col("center_lat"))) * cos(radians(col("lat_station"))) * pow(sin(radians(col("lon_station") - col("center_lon")) / 2), 2))))

In [62]:
df_sensors_stations.printSchema()

root
 |-- ID: string (nullable = true)
 |-- LINK_POINTS: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- BOROUGH: string (nullable = true)
 |-- lats: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- lons: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- center_lat: double (nullable = true)
 |-- center_lon: double (nullable = true)
 |-- STATION: string (nullable = true)
 |-- lat_station: double (nullable = true)
 |-- lon_station: double (nullable = true)
 |-- dist_km: double (nullable = true)



In [63]:
# get nearest weather station for each sensor

w = Window.partitionBy("ID").orderBy(col("dist_km").asc())
df_sensors_stations = df_sensors_stations \
    .withColumn("rn", row_number().over(w)) \
    .filter(col("rn") == 1) \
    .select("ID", "LINK_POINTS", "BOROUGH", col("STATION").alias("nearest_station"))

In [65]:
df_sensors_stations.write.mode("overwrite").parquet(f"{base_path}/data/driving_behavior/dim")

In [66]:
df_sensors_stations = spark.read.parquet(f"{base_path}/data/driving_behavior/dim")

In [67]:
# add nearest station to speed data
df_speeds = df_speeds \
    .join(df_sensors_stations.select("ID", col("nearest_station").alias("STATION")), on=("ID"), how=("left"))

df_speeds.printSchema()

root
 |-- ID: string (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- SPEED: string (nullable = true)
 |-- TRAVEL_TIME: string (nullable = true)
 |-- LINK_POINTS: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- BOROUGH: string (nullable = true)
 |-- TIME: string (nullable = true)
 |-- day_type: string (nullable = false)
 |-- STATION: string (nullable = true)



In [68]:
# extract hour and date from timestamp
df_speeds = df_speeds \
    .withColumn("hour", hour(col("DATE"))) \
    .withColumn("day", to_date(col("DATE")))

df_speeds.printSchema()

root
 |-- ID: string (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- SPEED: string (nullable = true)
 |-- TRAVEL_TIME: string (nullable = true)
 |-- LINK_POINTS: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- BOROUGH: string (nullable = true)
 |-- TIME: string (nullable = true)
 |-- day_type: string (nullable = false)
 |-- STATION: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day: date (nullable = true)



In [94]:
# join speed and weather data over nearest station, hour and day
df_driving = df_speeds \
    .join(df_weather.select("STATION", "hour", "day", "HourlyPrecipitation", "precipitation_type"), on=["STATION", "hour", "day"], how=("left"))

df_driving.printSchema()

root
 |-- STATION: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day: date (nullable = true)
 |-- ID: string (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- SPEED: string (nullable = true)
 |-- TRAVEL_TIME: string (nullable = true)
 |-- LINK_POINTS: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- BOROUGH: string (nullable = true)
 |-- TIME: string (nullable = true)
 |-- day_type: string (nullable = false)
 |-- HourlyPrecipitation: string (nullable = true)
 |-- precipitation_type: string (nullable = true)



In [95]:
df_driving.show(5)

+-----------+----+----------+---+-------------------+-----+-----------+--------------------+---------+--------+--------+-------------------+------------------+
|    STATION|hour|       day| ID|               DATE|SPEED|TRAVEL_TIME|         LINK_POINTS|  BOROUGH|    TIME|day_type|HourlyPrecipitation|precipitation_type|
+-----------+----+----------+---+-------------------+-----+-----------+--------------------+---------+--------+--------+-------------------+------------------+
|USW00014732|   1|2023-07-22|212|2023-07-22 01:28:06|52.81|        207|[40.78802,-73.790...|   Queens|01:25:00|     sat|                0.0|               dry|
|USW00014732|   1|2023-07-22|211|2023-07-22 01:28:06| 8.07|       1910|[40.78795,-73.790...|   Queens|01:25:00|     sat|                0.0|               dry|
|USW00094728|   1|2023-07-22|213|2023-07-22 01:28:05|19.26|        123|[40.8014104,-73.9...|Manhattan|01:25:00|     sat|                0.0|               dry|
|USW00014732|   1|2023-07-22|298|2023-07

In [96]:
df_driving = df_driving \
    .select("ID", "SPEED", "HourlyPrecipitation", "precipitation_type", "day_type", "hour") \
    .filter(col("HourlyPrecipitation").isNotNull())

In [97]:
df_driving.printSchema()

root
 |-- ID: string (nullable = true)
 |-- SPEED: string (nullable = true)
 |-- HourlyPrecipitation: string (nullable = true)
 |-- precipitation_type: string (nullable = true)
 |-- TIME: string (nullable = true)
 |-- day_type: string (nullable = false)
 |-- hour: integer (nullable = true)



In [98]:
df_driving.show(5)

+---+-----+-------------------+------------------+--------+--------+----+
| ID|SPEED|HourlyPrecipitation|precipitation_type|    TIME|day_type|hour|
+---+-----+-------------------+------------------+--------+--------+----+
|212|52.81|                0.0|               dry|01:25:00|     sat|   1|
|211| 8.07|                0.0|               dry|01:25:00|     sat|   1|
|213|19.26|                0.0|               dry|01:25:00|     sat|   1|
|298|45.36|                0.0|               dry|01:25:00|     sat|   1|
|205| 7.45|                0.0|               dry|01:25:00|     sat|   1|
+---+-----+-------------------+------------------+--------+--------+----+
only showing top 5 rows



In [99]:
# type casts
df_driving = df_driving \
    .withColumn("SPEED",  col("SPEED").cast("double")) \
    .withColumn("HourlyPrecipitation", when(col("HourlyPrecipitation") == "T", lit(0.005)).otherwise(col("HourlyPrecipitation").cast("double"))) \

df_driving.printSchema()

root
 |-- ID: string (nullable = true)
 |-- SPEED: double (nullable = true)
 |-- HourlyPrecipitation: double (nullable = true)
 |-- precipitation_type: string (nullable = true)
 |-- TIME: string (nullable = true)
 |-- day_type: string (nullable = false)
 |-- hour: integer (nullable = true)



In [101]:
# remove rows where precipitation type does not fit the precipitation amount
df_driving = df_driving \
    .filter(~((col("HourlyPrecipitation") > 0.0) & (col("precipitation_type") == "dry"))) \
    .filter(~((col("HourlyPrecipitation") == 0.0) & (col("precipitation_type") != "dry")))

In [23]:
# calculate baseline (median speed whith no precipitation per sensor, day_type, hour)
df_dry = df_driving \
    .groupBy("ID", "day_type", "hour") \
    .agg(expr("percentile_approx(speed, 0.5)").alias("baseline_speed"), count("*").alias("n_dry"))

In [24]:
df_dry.printSchema()

root
 |-- ID: string (nullable = true)
 |-- day_type: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- baseline_speed: double (nullable = true)
 |-- n_dry: long (nullable = false)



In [25]:
df_dry.show(5)

+---+--------+----+--------------+-----+
| ID|day_type|hour|baseline_speed|n_dry|
+---+--------+----+--------------+-----+
|  1|     sat|   4|          2.48|  672|
|  1|     sat|  12|          6.21|  696|
|  1|     sat|  15|          2.48|  684|
|  1|     sat|  20|          1.86|  655|
|  1|     sun|   0|          2.48|  684|
+---+--------+----+--------------+-----+
only showing top 5 rows



In [26]:
# add baseline speed
# remove rows with too less samples
# calculate relative speed

n = 30

df_driving = df_driving \
    .join(df_dry, on=["ID", "day_type", "hour"], how=("inner")) \
    .filter(col("n_dry") >= n) \
    .withColumn("rel_speed", col("SPEED") / col("baseline_speed"))

In [28]:
# categorise precipitation values in intensity categories
df_driving = df_driving \
    .withColumn("intensity", when(col("HourlyPrecipitation") == 0.0, None) \
                .when((col("precipitation_type") == "rain") & (col("HourlyPrecipitation") <= 0.005), "trace") \
                .when((col("precipitation_type") == "rain") & (col("HourlyPrecipitation") <= 2.5), "light") \
                .when((col("precipitation_type") == "rain") & (col("HourlyPrecipitation") <= 10.0), "moderate") \
                .when((col("precipitation_type") == "rain") & (col("HourlyPrecipitation") > 10.0), "heavy") \
                .when((col("precipitation_type") == "snow") & (col("HourlyPrecipitation") <= 0.005), "trace") \
                .when((col("precipitation_type") == "snow") & (col("HourlyPrecipitation") <= 2.0), "light") \
                .when((col("precipitation_type") == "snow") & (col("HourlyPrecipitation") <= 5.0), "moderate") \
                .when((col("precipitation_type") == "snow") & (col("HourlyPrecipitation") > 5.0), "heavy") \
                .otherwise(None))

In [29]:
df_driving.printSchema()

root
 |-- ID: string (nullable = true)
 |-- day_type: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- SPEED: double (nullable = true)
 |-- HourlyPrecipitation: double (nullable = true)
 |-- precipitation_type: string (nullable = true)
 |-- TIME: string (nullable = true)
 |-- baseline_speed: double (nullable = true)
 |-- n_dry: long (nullable = false)
 |-- rel_speed: double (nullable = true)
 |-- intensity: string (nullable = true)



In [30]:
# calculate median relative speed per precipitation type and intensity
df_driving = df_driving \
    .groupBy("precipitation_type", "intensity") \
    .agg(expr("percentile_approx(rel_speed, 0.5)").alias("median_rel"))

In [31]:
df_driving.show()

+------------------+---------+------------------+
|precipitation_type|intensity|        median_rel|
+------------------+---------+------------------+
|              rain|    light| 0.945591322603219|
|              snow|    heavy|0.5131300296484541|
|              rain|    heavy|0.9125262421273618|
|              snow| moderate|0.7976623874305423|
|              snow|    light|0.9080481036077706|
|               dry|     NULL|               1.0|
|              rain|    trace|0.9790444258172674|
|              rain| moderate|0.9135704351281542|
|              snow|    trace| 0.989153254023793|
+------------------+---------+------------------+



In [33]:
df_driving.write.mode("overwrite").parquet(f"{base_path}/data/driving_behavior")

In [38]:
spark.stop()